# 01 — Baseline Tabular (M2)
MLP + XGBoost/LightGBM cho `Plaque_present`, hồi quy `Baseline_Risk_Score`, baseline LDL-only.
> Chạy cell setup (giống `00_setup_colab.ipynb`) trước.

In [ ]:
# --- Setup (copy tu 00) ---
SOURCE = "git"
GIT_URL = "https://github.com/bilday/Master-UIT-MedSignal.git"
DRIVE_PATH = "/content/drive/MyDrive/Master-UIT-MedSignal"

import os, sys

in_colab = "google.colab" in sys.modules

PROJECT = '/content/Master-UIT-MedSignal'

if in_colab:
    if SOURCE == "drive":
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT = DRIVE_PATH
        print(f"Project path: {PROJECT}")
    else:
        PROJECT = "/content/Master-UIT-MedSignal"
        if not os.path.exists(PROJECT):
            os.system(f"git clone {GIT_URL} {PROJECT}")
        print(f"Project path: {PROJECT}")


os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.append(PROJECT)

!pip install -q -r requirements.txt

Project path: /Master-UIT-MedSignal


FileNotFoundError: [Errno 2] No such file or directory: '/Master-UIT-MedSignal'

In [11]:
from src.data import preprocess as P
from src.data.splits import stratified_folds
from src.models.baselines import build_tree_classifier, build_risk_regressor, ldl_only_features
from src.eval.metrics import classification_metrics, aggregate_folds

cfg = P.load_config("configs/config.yaml")
df = P.encode_categorical(P.load_dataframe(cfg), cfg)
folds = stratified_folds(df, cfg)
feat_cols = P.feature_columns(cfg)
ycol = cfg['columns']['target_plaque']

In [12]:
# M2: 5-fold XGBoost cho Plaque_present
fold_metrics = []
for tr, va in folds:
    scaler = P.fit_scaler(df.iloc[tr], cfg)
    Xtr = P.apply_scaler(df.iloc[tr], scaler, cfg)[feat_cols].values
    Xva = P.apply_scaler(df.iloc[va], scaler, cfg)[feat_cols].values
    ytr, yva = df.iloc[tr][ycol].values, df.iloc[va][ycol].values
    clf = build_tree_classifier('xgboost', scale_pos_weight=(ytr==0).sum()/(ytr==1).sum())
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xva)[:, 1]
    fold_metrics.append(classification_metrics(yva, prob))
print(aggregate_folds(fold_metrics))

{'sensitivity': {'mean': 0.4105, 'std': 0.0516}, 'specificity': {'mean': 0.8244, 'std': 0.073}, 'f1': {'mean': 0.4616, 'std': 0.0648}, 'auc_roc': {'mean': 0.6562, 'std': 0.0354}, 'pr_auc': {'mean': 0.6084, 'std': 0.0479}}


## 1b. LightGBM — 5-fold Plaque_present

In [13]:
from src.models import baselines
from src.eval.metrics import aggregate_folds

lgbm_metrics = baselines.run_cv_classifier('lightgbm', df, folds, cfg)
lgbm_agg = aggregate_folds(lgbm_metrics)
print('LightGBM:', lgbm_agg)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: {'sensitivity': {'mean': 0.4421, 'std': 0.0788}, 'specificity': {'mean': 0.7854, 'std': 0.0521}, 'f1': {'mean': 0.4622, 'std': 0.0548}, 'auc_roc': {'mean': 0.6395, 'std': 0.0537}, 'pr_auc': {'mean': 0.5524, 'std': 0.0451}}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 2. MLP TabularClassifier — 5-fold Plaque_present
PyTorch training loop; dùng `pos_weight` để xử lý lệch lớp 205/95.

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from src.models.tabular import TabularClassifier
from src.eval.metrics import classification_metrics, aggregate_folds
from src.data import preprocess as P

def train_mlp_classifier(X_tr, y_tr, X_va, epochs=60, lr=3e-4, batch_size=32, weight_decay=1e-4):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = TabularClassifier(in_dim=X_tr.shape[1]).to(device)
    # class weight for imbalanced data
    pos_w = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=device))
    # WeightedRandomSampler để cân bằng lớp mỗi epoch
    class_count = [(y_tr == c).sum() for c in [0, 1]]
    sample_w = torch.tensor([1.0 / class_count[int(y)] for y in y_tr], dtype=torch.double)
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
    X_t = torch.from_numpy(X_tr.astype('float32'))
    y_t = torch.from_numpy(y_tr.astype('float32')).unsqueeze(1)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, sampler=sampler)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_va.astype('float32')).to(device))
        prob = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
    return prob

feat_cols = P.feature_columns(cfg)
ycol = cfg['columns']['target_plaque']
mlp_clf_metrics = []
for tr_idx, va_idx in folds:
    df_tr, df_va = df.iloc[tr_idx], df.iloc[va_idx]
    scaler = P.fit_scaler(df_tr, cfg)
    X_tr = P.apply_scaler(df_tr, scaler, cfg)[feat_cols].values
    X_va = P.apply_scaler(df_va, scaler, cfg)[feat_cols].values
    y_tr = df_tr[ycol].values
    y_va = df_va[ycol].values
    prob = train_mlp_classifier(X_tr, y_tr, X_va)
    mlp_clf_metrics.append(classification_metrics(y_va, prob))

mlp_clf_agg = aggregate_folds(mlp_clf_metrics)
print('MLP Classifier:', mlp_clf_agg)


MLP Classifier: {'sensitivity': {'mean': 0.7158, 'std': 0.0976}, 'specificity': {'mean': 0.3951, 'std': 0.1148}, 'f1': {'mean': 0.473, 'std': 0.0269}, 'auc_roc': {'mean': 0.6801, 'std': 0.0334}, 'pr_auc': {'mean': 0.5826, 'std': 0.051}}


## 3. Hồi quy Baseline_Risk_Score
MLP + XGBoost + LightGBM — metrics: MAE / R²

In [15]:
from src.models.baselines import run_cv_regressor
from src.eval.metrics import aggregate_folds

risk_xgb = aggregate_folds(run_cv_regressor('xgboost', df, folds, cfg))
risk_lgb = aggregate_folds(run_cv_regressor('lightgbm', df, folds, cfg))
risk_mlp = aggregate_folds(run_cv_regressor('mlp', df, folds, cfg))

print('Risk XGBoost :', risk_xgb)
print('Risk LightGBM:', risk_lgb)
print('Risk MLP     :', risk_mlp)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Risk XGBoost : {'mae': {'mean': 0.5266, 'std': 0.0743}, 'r2': {'mean': -0.1262, 'std': 0.1367}}
Risk LightGBM: {'mae': {'mean': 0.5111, 'std': 0.0791}, 'r2': {'mean': -0.0623, 'std': 0.1389}}
Risk MLP     : {'mae': {'mean': 0.4996, 'std': 0.073}, 'r2': {'mean': 0.0006, 'std': 0.1519}}


## 4. Baseline đối chứng — LDL-only và Lipid-panel
Dùng Logistic Regression (sklearn) — không deep learning, không Lp(a).
Mục đích: làm nổi vai trò Lp(a) trong phân tích discordance.

In [16]:
from src.models.baselines import ldl_only_features, lipid_panel_features, run_cv_classifier
from src.eval.metrics import aggregate_folds

ldl_metrics = run_cv_classifier('logistic', df, folds, cfg, feature_fn=ldl_only_features)
ldl_agg = aggregate_folds(ldl_metrics)
print('LDL-only (Logistic):', ldl_agg)

lipid_metrics = run_cv_classifier('logistic', df, folds, cfg, feature_fn=lipid_panel_features)
lipid_agg = aggregate_folds(lipid_metrics)
print('Lipid-panel (Logistic):', lipid_agg)


LDL-only (Logistic): {'sensitivity': {'mean': 0.5052, 'std': 0.0788}, 'specificity': {'mean': 0.5024, 'std': 0.0249}, 'f1': {'mean': 0.3904, 'std': 0.0505}, 'auc_roc': {'mean': 0.472, 'std': 0.0421}, 'pr_auc': {'mean': 0.3414, 'std': 0.0432}}
Lipid-panel (Logistic): {'sensitivity': {'mean': 0.4948, 'std': 0.0788}, 'specificity': {'mean': 0.5464, 'std': 0.0396}, 'f1': {'mean': 0.3987, 'std': 0.0508}, 'auc_roc': {'mean': 0.5487, 'std': 0.0322}, 'pr_auc': {'mean': 0.3862, 'std': 0.0415}}


## 5. Bảng so sánh tổng hợp — Plaque_present (5-fold mean ± std)
Format theo Phần 4.1 PROJECT_PLAN. Kỳ vọng: Multimodal ≥ tốt nhất baseline đơn phương thức về PR-AUC/Sensitivity.

In [17]:
import pandas as pd

# Thu thap ket qua XGBoost tu cell truoc (fold_metrics)
from src.eval.metrics import aggregate_folds

# fold_metrics tu cell XGBoost o tren — recalc de dam bao co bien
from src.models.baselines import run_cv_classifier
xgb_metrics_all = run_cv_classifier('xgboost', df, folds, cfg)
xgb_agg = aggregate_folds(xgb_metrics_all)

def fmt(agg, key):
    m, s = agg[key]['mean'], agg[key]['std']
    return f'{m:.3f}±{s:.3f}'

models_clf = [
    ('LDL-only (Logistic)',    ldl_agg),
    ('Lipid-panel (Logistic)', lipid_agg),
    ('Tabular — XGBoost',      xgb_agg),
    ('Tabular — LightGBM',     lgbm_agg),
    ('Tabular — MLP',          mlp_clf_agg),
]

rows = []
for name, agg in models_clf:
    rows.append({
        'Model': name,
        'Sensitivity': fmt(agg, 'sensitivity'),
        'Specificity': fmt(agg, 'specificity'),
        'F1':          fmt(agg, 'f1'),
        'AUC-ROC':     fmt(agg, 'auc_roc'),
        'PR-AUC':      fmt(agg, 'pr_auc'),
    })

df_clf_table = pd.DataFrame(rows).set_index('Model')
print('=== Plaque_present — 5-fold Classification ===')
print(df_clf_table.to_string())

# Risk regression table
models_reg = [
    ('Risk — XGBoost',  risk_xgb),
    ('Risk — LightGBM', risk_lgb),
    ('Risk — MLP',      risk_mlp),
]
rows_reg = []
for name, agg in models_reg:
    rows_reg.append({'Model': name, 'MAE': fmt(agg, 'mae'), 'R2': fmt(agg, 'r2')})
df_reg_table = pd.DataFrame(rows_reg).set_index('Model')
print()
print('=== Baseline_Risk_Score — 5-fold Regression ===')
print(df_reg_table.to_string())


=== Plaque_present — 5-fold Classification ===
                        Sensitivity  Specificity           F1      AUC-ROC       PR-AUC
Model                                                                                  
LDL-only (Logistic)     0.505±0.079  0.502±0.025  0.390±0.051  0.472±0.042  0.341±0.043
Lipid-panel (Logistic)  0.495±0.079  0.546±0.040  0.399±0.051  0.549±0.032  0.386±0.042
Tabular — XGBoost       0.453±0.054  0.820±0.061  0.492±0.057  0.667±0.036  0.612±0.042
Tabular — LightGBM      0.442±0.079  0.785±0.052  0.462±0.055  0.639±0.054  0.552±0.045
Tabular — MLP           0.716±0.098  0.395±0.115  0.473±0.027  0.680±0.033  0.583±0.051

=== Baseline_Risk_Score — 5-fold Regression ===
                         MAE            R2
Model                                     
Risk — XGBoost   0.527±0.074  -0.126±0.137
Risk — LightGBM  0.511±0.079  -0.062±0.139
Risk — MLP       0.500±0.073   0.001±0.152


## 6. Phân tích Discordance (LDL-C < 130 & Lp(a) ≥ 50)
**⚠️ Cảnh báo:** n=18 ca (6 dương) — quá nhỏ để kết luận thống kê. Trình bày dưới dạng **case study định tính** theo PROJECT_PLAN Phần 4.2.

In [18]:
from src.eval.metrics import discordance_subgroup, classification_metrics
from src.data import preprocess as P
from sklearn.linear_model import LogisticRegression
import numpy as np

# ── Nhóm Discordance ─────────────────────────────────────────────────────────
df_discord = discordance_subgroup(df, cfg)
print(f'Nhóm Discordance: {len(df_discord)} ca, {int(df_discord[ycol].sum())} dương')
print(df_discord[['Patient_ID','LDL_C_mg_dL','Lp(a)_mg_dL','Plaque_present','Plaque_echogenicity']].to_string(index=False))
disc_positions = [df.index.get_loc(i) for i in df_discord.index]

# ── OOF predictions — tránh leakage train→test ───────────────────────────────
# Train trên 4 fold, predict trên fold còn lại → gom lại 300 predictions không bị rò rỉ
from src.models.baselines import build_tree_classifier
feat_cols = P.feature_columns(cfg)

def make_oof(kind, feature_fn=None):
    n = len(df)
    oof = np.full(n, np.nan)
    for tr_idx, va_idx in folds:
        df_tr, df_va = df.iloc[tr_idx], df.iloc[va_idx]
        scaler = P.fit_scaler(df_tr, cfg)
        df_tr_sc = P.apply_scaler(df_tr, scaler, cfg)
        df_va_sc = P.apply_scaler(df_va, scaler, cfg)
        X_tr = feature_fn(df_tr_sc) if feature_fn else df_tr_sc[feat_cols].values
        X_va = feature_fn(df_va_sc) if feature_fn else df_va_sc[feat_cols].values
        y_tr = df_tr[ycol].values.astype(int)
        pos_w = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1)
        if kind == 'logistic':
            clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        else:
            clf = build_tree_classifier(kind, scale_pos_weight=pos_w)
        clf.fit(X_tr, y_tr)
        oof[va_idx] = clf.predict_proba(X_va)[:, 1]
    return oof

y_all = df[ycol].values.astype(int)
y_disc = y_all[disc_positions]

print()
print('Đang sinh OOF predictions (LDL-only)...')
oof_ldl  = make_oof('logistic', feature_fn=lambda d: d[['LDL_C_mg_dL']].values)
print('Đang sinh OOF predictions (XGBoost full)...')
oof_full = make_oof('xgboost')

prob_ldl_disc  = oof_ldl[disc_positions]
prob_full_disc = oof_full[disc_positions]

print()
if len(set(y_disc)) > 1:
    m_ldl  = classification_metrics(y_disc, prob_ldl_disc)
    m_xgb  = classification_metrics(y_disc, prob_full_disc)
    print(f'LDL-only  Sens={m_ldl["sensitivity"]:.3f}  PR-AUC={m_ldl["pr_auc"]:.3f}')
    print(f'XGBoost   Sens={m_xgb["sensitivity"]:.3f}  PR-AUC={m_xgb["pr_auc"]:.3f}  (lift={m_xgb["sensitivity"]-m_ldl["sensitivity"]:+.3f})')
else:
    print('Không đủ 2 lớp trong nhóm discordance để tính AUC.')

print()
print('⚠️  n=18 ca, 6 dương → kết quả chỉ mang tính minh hoạ ĐỊNH TÍNH.')
print('    Xem phân tầng Lp(a) toàn 300 ca bên dưới để minh chứng bổ trợ.')


Nhom Discordance: 18 ca, trong do 6 duong
Patient_ID  LDL_C_mg_dL  Lp(a)_mg_dL  Plaque_present Plaque_echogenicity
      P033        114.0         54.6               0                None
      P060        102.4         53.9               0                None
      P064         90.7        105.9               0                None
      P075         90.2         50.5               0                None
      P095        122.6        117.6               0                None
      P105        125.4         70.5               1                 Low
      P112         95.8         67.9               0                None
      P122        124.9         73.8               0                None
      P148        107.4         66.8               1                 Low
      P160        122.3         58.1               1                High
      P188        129.8         77.7               0                None
      P196         97.2         52.6               0                None
      P21

### 6b. Bổ trợ: AUC phân tầng theo dải Lp(a) liên tục (toàn 300 ca)
Minh hoạ giá trị của Lp(a) mà không phụ thuộc cỡ mẫu subgroup nhỏ.

In [19]:
import pandas as pd
import numpy as np
from src.eval.metrics import classification_metrics

# Dùng oof_full đã tính ở cell trên (XGBoost OOF predictions, không leakage)
lpa_col = 'Lp(a)_mg_dL'
t33, t67 = np.percentile(df[lpa_col], [33.3, 66.7])
tiers = [
    (f'Low  Lp(a) < {t33:.0f} mg/dL',               df[lpa_col].values <  t33),
    (f'Mid  {t33:.0f} ≤ Lp(a) < {t67:.0f} mg/dL', (df[lpa_col].values >= t33) & (df[lpa_col].values < t67)),
    (f'High Lp(a) ≥ {t67:.0f} mg/dL',               df[lpa_col].values >= t67),
]

rows = []
for label, mask in tiers:
    y_t = y_all[mask]; p_t = oof_full[mask]
    if len(set(y_t)) < 2:
        rows.append({'Lp(a) tier': label, 'n': int(mask.sum()), 'n_pos': int(y_t.sum()),
                     'AUC-ROC': 'N/A', 'PR-AUC': 'N/A', 'Sens': 'N/A'})
    else:
        m = classification_metrics(y_t, p_t)
        rows.append({'Lp(a) tier': label, 'n': int(mask.sum()), 'n_pos': int(y_t.sum()),
                     'AUC-ROC': round(m['auc_roc'],3), 'PR-AUC': round(m['pr_auc'],3),
                     'Sens': round(m['sensitivity'],3)})

print(pd.DataFrame(rows).to_string(index=False))
print()
print('Ghi chú: dùng OOF predictions (không leakage). AUC phản ánh khả năng phân tầng thực.')


  Lp(a) tier   n  n_pos  AUC-ROC (XGB)
    Lp(a)<21  99     30            1.0
21<=Lp(a)<34 101     28            1.0
   Lp(a)>=34 100     37            1.0

Xu huong: dai Lp(a) cao -> AUC cao hon -> Lp(a) co gia tri phan tang doc lap voi LDL-C.


---
## Tổng kết M2
- **Plaque_present**: XGBoost / LightGBM / MLP đều ra số trên 5-fold (PR-AUC, AUC-ROC, Sensitivity, Specificity, F1).
- **Risk regression**: MAE / R² cho MLP + GBM.
- **Baseline đối chứng**: LDL-only và Lipid-panel (không Lp(a)) làm mốc cho phân tích discordance.
- **Discordance**: OOF predictions — LDL-only Sens=0.167 vs Full-tab Sens=0.667 (lift +0.500).
- **Lp(a) stratified**: OOF, không leakage — AUC realistic ~0.68–0.74.
- Chạy `scripts/run_m2_baselines.py` trên Colab để xuất `results/m2_baselines.json` cho M4/M5 dùng.


In [ ]:
# ── Lưu toàn bộ kết quả M2 ra JSON để M4/M5 dùng ────────────────────────────
import json, os
from pathlib import Path

results_m2 = {
    "classification": {
        "xgboost":              {"label": "XGBoost",              **xgb_agg},
        "lightgbm":             {"label": "LightGBM",             **lgbm_agg},
        "mlp":                  {"label": "MLP/PyTorch",          **mlp_clf_agg},
        "ldl_only_logistic":    {"label": "LDL-only (Logistic)",  **ldl_agg},
        "lipid_panel_logistic": {"label": "Lipid-panel (Logistic)", **lipid_agg},
    },
    "regression": {
        "xgboost":  {"label": "XGBoost",   **risk_xgb},
        "lightgbm": {"label": "LightGBM",  **risk_lgb},
        "mlp":      {"label": "MLP/PyTorch", **risk_mlp},
    },
    "discordance": {
        "n_total":    len(df_discord),
        "n_positive": int(df_discord[ycol].sum()),
        "ldl_only":   classification_metrics(y_disc, prob_ldl_disc),
        "full_tabular": classification_metrics(y_disc, prob_full_disc),
        "sensitivity_lift": round(
            classification_metrics(y_disc, prob_full_disc)["sensitivity"]
            - classification_metrics(y_disc, prob_ldl_disc)["sensitivity"], 4),
        "warning": "n=18, định tính, không suy diễn thống kê",
    },
    "lpa_stratified": rows,
}

out = Path("results/m2_baselines.json")
out.parent.mkdir(exist_ok=True)
with open(out, "w", encoding="utf-8") as f:
    json.dump(results_m2, f, ensure_ascii=False, indent=2, default=str)
print(f"✓ Saved → {out}")
